# 부두별 선석별 평균선석점유율
- 선석 점유율 = Σ 작업시간 ÷ (선석 수 × 기간 시간) × 100
- 작업시간 = 접안-출항

## 자료없음
- 2부두 9선석:2022~2025년 접안, 출항 기록 없음
- 4부두 4선석:4선석의 접안, 출항 기록 없음

# 부두별 선석별 총작업시간

In [19]:
import pandas as pd
import plotly.express as px

# ======================================
# 1. CSV 불러오기
# ======================================

def load_dataset(filename):
    BASE_DIR = r'C:\work\Final_Project\data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'

    try:
        return pd.read_csv(TARGET_PATH, encoding='utf-8-sig')
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {TARGET_PATH}")
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='cp949')


port1 = load_dataset('port1')
port2 = load_dataset('port2')
port3 = load_dataset('port3')
port4 = load_dataset('port4')
port5 = load_dataset('port5')

ports = [
    (1, port1),
    (2, port2),
    (3, port3),
    (4, port4),
    (5, port5),
]

# 분석 기간
START = pd.Timestamp("2022-01-01")
END = pd.Timestamp("2026-01-01")


# ======================================
# 2. 부두별 작업시간 계산
# ======================================

frames = []

for number, port in ports:

    if port is None:
        print(f"{number}부두: 파일이 없어 제외")
        continue

    data = port.copy()

    # 접안시각 변환
    data["접안시각"] = pd.to_datetime(
        data["접안(예정)일시"],
        format="%Y-%m-%d %H:%M",
        errors="coerce"
    )

    # 출항시각 변환
    data["출항시각"] = pd.to_datetime(
        data["출항(예정)일시"],
        format="%Y-%m-%d %H:%M",
        errors="coerce"
    )

    # 오류 및 누락 데이터 제외
    invalid = (
        data[["접안시각", "출항시각"]].isna().any(axis=1)
        | (data["출항시각"] < data["접안시각"])
    )

    if invalid.any():
        print(f"{number}부두: 오류·누락 {invalid.sum()}건 제외")

    data = data.loc[~invalid].copy()

    # 2022~2025년과 겹치는 기록만 선택
    data = data.loc[
        (data["접안시각"] < END)
        & (data["출항시각"] > START)
    ].copy()

    # 분석 기간 내 작업시간만 반영
    work_start = data["접안시각"].clip(lower=START)
    work_end = data["출항시각"].clip(upper=END)

    # 작업시간 계산 (시간 단위)
    data["작업시간_시간"] = (
        work_end - work_start
    ).dt.total_seconds() / 3600

    data["부두"] = f"{number}부두"

    frames.append(
        data[["부두", "작업시간_시간"]]
    )


# ======================================
# 3. 부두별 총 작업시간 합산
# ======================================

detail = pd.concat(frames, ignore_index=True)

result = (
    detail.groupby("부두")
    .agg(
        기록수=("작업시간_시간", "count"),
        총작업시간_시간=("작업시간_시간", "sum")
    )
    .reset_index()
)

# 부두 순서 지정
port_order = [
    "1부두",
    "2부두",
    "3부두",
    "4부두",
    "5부두"
]

result["부두"] = pd.Categorical(
    result["부두"],
    categories=port_order,
    ordered=True
)

result = result.sort_values("부두")

print("\n2022~2025년 부두별 총 작업시간")
print(result.round(2).to_string(index=False))


# ======================================
# 4. Plotly 막대그래프
# ======================================

port_colors = {
    "1부두": "#4C78A8",
    "2부두": "#F58518",
    "3부두": "#54A24B",
    "4부두": "#E45756",
    "5부두": "#B279A2"
}

fig = px.bar(
    result,
    x="부두",
    y="총작업시간_시간",
    color="부두",
    text="총작업시간_시간",
    color_discrete_map=port_colors,
    category_orders={"부두": port_order},
    custom_data=["기록수"]
)

fig.update_traces(
    texttemplate="%{y:,.0f}",
    textposition="outside",
    hovertemplate=(
        "부두: %{x}<br>"
        "총 작업시간: %{y:,.1f}시간<br>"
        "기록 수: %{customdata[0]:,.0f}건"
        "<extra></extra>"
    )
)


# ======================================
# 5. 그래프 디자인
# ======================================

maximum = result["총작업시간_시간"].max()

fig.update_layout(
    title=dict(
        text="2022~2025년 부산 신항 부두별 총 작업시간",
        x=0.5,
        xanchor="center",
        font=dict(size=22)
    ),

    xaxis=dict(
        title="부두",
        showgrid=False
    ),

    yaxis=dict(
        title="총 작업시간 (시간)",
        range=[0, maximum * 1.15] if maximum > 0 else [0, 1],
        tickformat=",",
        showgrid=True,
        gridcolor="#E5E7EB"
    ),

    template="plotly_white",

    width=1100,
    height=600,

    showlegend=False,

    bargap=0.35,

    margin=dict(
        t=100,
        b=70,
        l=80,
        r=40
    )
)


# ======================================
# 6. 그래프 출력
# ======================================

fig.show()

4부두: 오류·누락 1건 제외

2022~2025년 부두별 총 작업시간
 부두  기록수  총작업시간_시간
1부두 2928  82679.28
2부두 6726 153601.67
3부두 3772  82095.98
4부두 3419  74595.18
5부두 3872  92010.67


# 선석수

In [20]:
import pandas as pd

# 앞서 제공한 이미지 기준 선석 수
선석수 = {1: 3, 2: 6, 3: 4, 4: 4, 5: 4}

선석수표 = pd.DataFrame({
    "부두": list(선석수.keys()),
    "선석수": list(선석수.values())
})

print(선석수표.to_string(index=False))

 부두  선석수
  1    3
  2    6
  3    4
  4    4
  5    4


# 부두별 선석별 평균 선석점유율
- 해당선석의 시간 내 총 작업시간/35,064시간(22~25년 전체 시간) * 100

In [21]:
import pandas as pd
import plotly.graph_objects as go

# ======================================
# 1. CSV 불러오기
# ======================================

def load_dataset(filename):
    BASE_DIR = r'C:\work\Final_Project\data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'

    try:
        return pd.read_csv(TARGET_PATH, encoding='utf-8-sig')
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {TARGET_PATH}")
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='cp949')


# 부두별 CSV 불러오기
port1 = load_dataset('port1')
port2 = load_dataset('port2')
port3 = load_dataset('port3')
port4 = load_dataset('port4')
port5 = load_dataset('port5')

# 부두 번호와 데이터 연결
ports = [
    (1, port1),
    (2, port2),
    (3, port3),
    (4, port4),
    (5, port5)
]

# 부두별 선석 번호
expected_berths = {
    1: [1, 2, 3],
    2: [4, 5, 6, 7, 8, 9],
    3: [1, 2, 3, 4],
    4: [1, 2, 3, 4],
    5: [1, 2, 3, 4]
}

# 분석 기간: 2022~2025년
START = pd.Timestamp("2022-01-01")
END = pd.Timestamp("2026-01-01")

# 분석 기간 총 시간
period_hours = (END - START).total_seconds() / 3600


# ======================================
# 2. 부두별 작업시간 계산
# ======================================

frames = []

for number, port in ports:

    if port is None:
        raise FileNotFoundError(
            f"{number}부두 CSV 파일을 확인하세요."
        )

    data = port.copy()
    data.columns = data.columns.str.strip()

    # 접안시각 변환
    data["접안시각"] = pd.to_datetime(
        data["접안(예정)일시"],
        errors="coerce"
    )

    # 출항시각 변환
    data["출항시각"] = pd.to_datetime(
        data["출항(예정)일시"],
        errors="coerce"
    )

    # 선석 번호 변환
    data["선석"] = pd.to_numeric(
        data["선석"],
        errors="coerce"
    )

    # 오류 및 누락 데이터 제외
    invalid = (
        data[["접안시각", "출항시각", "선석"]]
        .isna().any(axis=1)
        | (data["출항시각"] < data["접안시각"])
    )

    if invalid.any():
        print(f"{number}부두: 오류·누락 {invalid.sum()}건 제외")

    data = data.loc[~invalid].copy()

    # 해당 부두의 선석만 선택
    data = data[
        data["선석"].isin(expected_berths[number])
    ].copy()

    # 분석 기간과 겹치는 기록 선택
    data = data.loc[
        (data["접안시각"] < END)
        & (data["출항시각"] > START)
    ].copy()

    # 분석 기간에 포함되는 시간만 계산
    work_start = data["접안시각"].clip(lower=START)
    work_end = data["출항시각"].clip(upper=END)

    # 작업시간 = 출항시각 - 접안시각
    data["작업시간_시간"] = (
        work_end - work_start
    ).dt.total_seconds() / 3600

    data["부두"] = number

    frames.append(
        data[["부두", "작업시간_시간"]]
    )

detail = pd.concat(frames, ignore_index=True)


# ======================================
# 3. 부두별 평균 선석점유율 계산
# ======================================

# 부두별 총 작업시간 합산
result = (
    detail.groupby("부두")
    .agg(
        기록수=("작업시간_시간", "count"),
        총작업시간_시간=("작업시간_시간", "sum")
    )
    .reset_index()
)

# 모든 부두가 표시되도록 설정
result = (
    result.set_index("부두")
    .reindex([1, 2, 3, 4, 5])
    .reset_index()
)

# 선석 수
berth_counts = {
    number: len(berths)
    for number, berths in expected_berths.items()
}

result["선석수"] = result["부두"].map(berth_counts)

# 평균 선석점유율 계산
result["평균선석점유율_퍼센트"] = (
    result["총작업시간_시간"]
    / (result["선석수"] * period_hours)
    * 100
)

# 부두 이름
result["부두명"] = result["부두"].astype(str) + "부두"

print("\n[2022~2025년 부산 신항 부두별 평균 선석점유율]")

print(
    result[
        [
            "부두명",
            "선석수",
            "기록수",
            "총작업시간_시간",
            "평균선석점유율_퍼센트"
        ]
    ].round(2).to_string(index=False, na_rep="자료 없음")
)


# ======================================
# 4. Plotly 막대그래프
# ======================================

port_colors = {
    1: "#4C78A8",
    2: "#F58518",
    3: "#54A24B",
    4: "#E45756",
    5: "#B279A2"
}

fig = go.Figure()

for _, row in result.iterrows():

    number = int(row["부두"])
    value = row["평균선석점유율_퍼센트"]

    fig.add_trace(
        go.Bar(
            x=[f"{number}부두"],
            y=[value],
            name=f"{number}부두",
            marker_color=port_colors[number],

            text=(
                f"{value:.2f}%"
                if pd.notna(value)
                else "자료 없음"
            ),
            textposition="outside",

            customdata=[[
                row["총작업시간_시간"],
                row["선석수"],
                row["기록수"]
            ]],

            hovertemplate=(
                "부두: %{x}<br>"
                "평균 선석점유율: %{y:.2f}%<br>"
                "총 작업시간: %{customdata[0]:,.1f}시간<br>"
                "선석 수: %{customdata[1]:.0f}개<br>"
                "기록 수: %{customdata[2]:,.0f}건"
                "<extra></extra>"
            )
        )
    )


# ======================================
# 5. 그래프 디자인
# ======================================

maximum = result["평균선석점유율_퍼센트"].max()

upper = (
    max(100, maximum * 1.15)
    if pd.notna(maximum)
    else 100
)

fig.update_layout(

    title=dict(
        text="2022~2025년 부산 신항 부두별 평균 선석점유율",
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),

    xaxis=dict(
        title="부두",
        categoryorder="array",
        categoryarray=[
            "1부두",
            "2부두",
            "3부두",
            "4부두",
            "5부두"
        ],
        showgrid=False
    ),

    yaxis=dict(
        title="평균 선석점유율 (%)",
        range=[0, upper],
        ticksuffix="%",
        showgrid=True,
        gridcolor="#E5E7EB"
    ),

    barmode="group",
    template="plotly_white",

    width=1100,
    height=600,

    bargap=0.35,
    showlegend=False,

    margin=dict(
        t=100,
        b=70,
        l=80,
        r=40
    )
)


# ======================================
# 6. PyCharm 주피터 노트북 출력
# ======================================

fig.show()

4부두: 오류·누락 1건 제외

[2022~2025년 부산 신항 부두별 평균 선석점유율]
부두명  선석수  기록수  총작업시간_시간  평균선석점유율_퍼센트
1부두    3 2928  82679.28        78.60
2부두    6 6726 153601.67        73.01
3부두    4 3772  82095.98        58.53
4부두    4 3419  74595.18        53.19
5부두    4 3872  92010.67        65.60


# 부두별 선석별 총 작업시간

In [22]:
import pandas as pd
import plotly.graph_objects as go

# ======================================
# 1. CSV 불러오기
# ======================================

def load_dataset(filename):
    BASE_DIR = r'C:\work\Final_Project\data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'

    try:
        return pd.read_csv(TARGET_PATH, encoding='utf-8-sig')
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {TARGET_PATH}")
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='cp949')


# 부두별 CSV 불러오기
port1 = load_dataset('port1')
port2 = load_dataset('port2')
port3 = load_dataset('port3')
port4 = load_dataset('port4')
port5 = load_dataset('port5')


# 부두 번호와 데이터 연결
ports = [
    (1, port1),
    (2, port2),
    (3, port3),
    (4, port4),
    (5, port5)
]


# 분석 기간 설정
START = pd.Timestamp("2022-01-01")
END = pd.Timestamp("2026-01-01")

# 부두별 선석 번호
expected_berths = {
    1: [1, 2, 3],
    2: [4, 5, 6, 7, 8, 9],
    3: [1, 2, 3, 4],
    4: [1, 2, 3, 4],
    5: [1, 2, 3, 4],
}

# ======================================
# 2. 작업시간 계산
# ======================================

frames = []

for number, port in ports:
    data = port.copy()

    data["접안시각"] = pd.to_datetime(
        data["접안(예정)일시"],
        format="%Y-%m-%d %H:%M",
        errors="raise",
    )

    data["출항시각"] = pd.to_datetime(
        data["출항(예정)일시"],
        format="%Y-%m-%d %H:%M",
        errors="raise",
    )

    # 시각·선석 누락 또는 출항이 접안보다 빠른 기록 제외
    invalid = (
        data[["접안시각", "출항시각", "선석"]].isna().any(axis=1)
        | (data["출항시각"] < data["접안시각"])
    )

    if invalid.any():
        print(f"{number}부두: 오류·누락 {invalid.sum()}건 제외")

    data = data.loc[~invalid].copy()

    # 분석 기간과 겹치는 기록 선택
    data = data.loc[
        (data["접안시각"] < END)
        & (data["출항시각"] > START)
    ].copy()

    # 2022~2025년에 포함되는 구간만 반영
    work_start = data["접안시각"].clip(lower=START)
    work_end = data["출항시각"].clip(upper=END)

    # 작업시간 = 출항시각 - 접안시각
    data["작업시간_시간"] = (
        work_end - work_start
    ).dt.total_seconds() / 3600

    data["부두"] = number

    frames.append(
        data[["부두", "선석", "작업시간_시간"]]
    )

detail = pd.concat(frames, ignore_index=True)

# ======================================
# 3. 부두별·선석별 총 작업시간
# ======================================

result = (
    detail.groupby(["부두", "선석"])
    .agg(
        기록수=("작업시간_시간", "count"),
        총작업시간_시간=("작업시간_시간", "sum"),
    )
)

# 기록이 없는 선석도 표시하되 0시간으로 단정하지 않음
expected_index = pd.MultiIndex.from_tuples(
    [
        (number, berth)
        for number, berths in expected_berths.items()
        for berth in berths
    ],
    names=["부두", "선석"],
)

result = result.reindex(expected_index).reset_index()
result["기록수"] = result["기록수"].fillna(0).astype(int)

print("\n2022~2025년 부두별·선석별 총 작업시간")
print(result.round(2).to_string(index=False, na_rep="자료 없음"))

# ======================================
# 4. Plotly 바그래프
# ======================================

port_colors = {
    1: "#4C78A8",
    2: "#F58518",
    3: "#54A24B",
    4: "#E45756",
    5: "#B279A2",
}

fig = go.Figure()

for number in expected_berths:

    temp = result[result["부두"] == number].copy()

    labels = (
        temp["부두"].astype(str) + "부두 "
        + temp["선석"].astype(int).astype(str) + "선석"
    )

    values = temp["총작업시간_시간"]

    fig.add_trace(
        go.Bar(
            x=labels,
            y=values,
            name=f"{number}부두",
            marker_color=port_colors[number],
            text=[
                f"{v:,.1f}" if pd.notna(v) else "자료 없음"
                for v in values
            ],
            textposition="outside",
            customdata=temp[["기록수"]].to_numpy(),
            hovertemplate=(
                "선석: %{x}<br>"
                "총 작업시간: %{y:,.1f}시간<br>"
                "기록 수: %{customdata[0]:,.0f}건"
                "<extra></extra>"
            )
        )
    )

# ======================================
# 5. 그래프 디자인
# ======================================

maximum = result["총작업시간_시간"].max()
upper = max(maximum * 1.15, 1) if pd.notna(maximum) else 1

fig.update_layout(
    title=dict(
        text="2022~2025년 부두별·선석별 총 작업시간",
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    ),

    xaxis=dict(
        title="부두 / 선석",
        tickangle=-45,
        showgrid=False
    ),

    yaxis=dict(
        title="총 작업시간(시간)",
        range=[0, upper],
        tickformat=",",
        showgrid=True,
        gridcolor="#E5E7EB"
    ),

    barmode="group",
    template="plotly_white",
    width=1400,
    height=650,
    bargap=0.25,

    legend=dict(
        title="부두",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),

    margin=dict(
        t=110,
        b=100,
        l=70,
        r=30
    )
)

# ======================================
# 6. PyCharm 주피터 노트북 내부 출력
# ======================================

fig.show()

4부두: 오류·누락 1건 제외

2022~2025년 부두별·선석별 총 작업시간
 부두  선석  기록수  총작업시간_시간
  1   1 1110  25490.57
  1   2  975  27742.68
  1   3  843  29446.03
  2   4 1710  29982.72
  2   5 1536  30264.97
  2   6 1303  30925.38
  2   7 1136  31080.73
  2   8 1041  31347.87
  2   9    0     자료 없음
  3   1 1244  29537.48
  3   2 1332  21962.97
  3   3  740  21834.08
  3   4  456   8761.45
  4   1 1069  20503.68
  4   2 1135  26613.38
  4   3 1215  27478.12
  4   4    0     자료 없음
  5   1 1362  29422.40
  5   2 1074  27962.50
  5   3  372   7677.90
  5   4 1064  26947.87


# 부두별 선석별 평균선석점유율

In [23]:
import pandas as pd
import plotly.graph_objects as go

# ======================================
# 1. CSV 불러오기
# ======================================

def load_dataset(filename):
    BASE_DIR = r'C:\work\Final_Project\data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'

    try:
        return pd.read_csv(TARGET_PATH, encoding='utf-8-sig')
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {TARGET_PATH}")
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='cp949')


# 부두별 CSV 불러오기
port1 = load_dataset('port1')
port2 = load_dataset('port2')
port3 = load_dataset('port3')
port4 = load_dataset('port4')
port5 = load_dataset('port5')

ports = [
    (1, port1),
    (2, port2),
    (3, port3),
    (4, port4),
    (5, port5)
]

expected_berths = {
    1: [1, 2, 3],
    2: [4, 5, 6, 7, 8, 9],
    3: [1, 2, 3, 4],
    4: [1, 2, 3, 4],
    5: [1, 2, 3, 4]
}

START = pd.Timestamp("2022-01-01")
END = pd.Timestamp("2026-01-01")

period_hours = (END - START).total_seconds() / 3600

# ======================================
# 2. 작업시간 계산
# ======================================

frames = []

for number, port in ports:

    data = port.copy()
    data.columns = data.columns.str.strip()

    data["접안시각"] = pd.to_datetime(
        data["접안(예정)일시"],
        errors="coerce"
    )

    data["출항시각"] = pd.to_datetime(
        data["출항(예정)일시"],
        errors="coerce"
    )

    data["선석"] = pd.to_numeric(
        data["선석"], errors="coerce"
    )

    invalid = (
        data[["접안시각", "출항시각", "선석"]].isna().any(axis=1)
        | (data["출항시각"] < data["접안시각"])
    )

    if invalid.any():
        print(f"{number}부두: 오류·누락 {invalid.sum()}건 제외")

    data = data.loc[~invalid].copy()

    data = data[
        data["선석"].isin(expected_berths[number])
    ].copy()

    data = data.loc[
        (data["접안시각"] < END)
        & (data["출항시각"] > START)
    ].copy()

    work_start = data["접안시각"].clip(lower=START)
    work_end = data["출항시각"].clip(upper=END)

    data["작업시간_시간"] = (
        work_end - work_start
    ).dt.total_seconds() / 3600

    data["부두"] = number

    frames.append(
        data[["부두", "선석", "작업시간_시간"]]
    )

detail = pd.concat(frames, ignore_index=True)

# ======================================
# 3. 선석점유율 계산
# ======================================

result = (
    detail.groupby(["부두", "선석"])
    .agg(총작업시간_시간=("작업시간_시간", "sum"))
)

expected_index = pd.MultiIndex.from_tuples(
    [
        (number, berth)
        for number, berths in expected_berths.items()
        for berth in berths
    ],
    names=["부두", "선석"]
)

result = result.reindex(expected_index).reset_index()

result["평균선석점유율_퍼센트"] = (
    result["총작업시간_시간"] / period_hours * 100
)

print("\n[2022~2025년 부두별·선석별 평균 선석점유율]")
print(result.round(2).to_string(index=False, na_rep="자료 없음"))

# ======================================
# 4. Plotly 바그래프
# ======================================

port_colors = {
    1: "#4C78A8",
    2: "#F58518",
    3: "#54A24B",
    4: "#E45756",
    5: "#B279A2"
}

fig = go.Figure()

for number in expected_berths:

    temp = result[result["부두"] == number].copy()

    labels = (
        temp["부두"].astype(str) + "부두 "
        + temp["선석"].astype(int).astype(str) + "선석"
    )

    values = temp["평균선석점유율_퍼센트"]

    fig.add_trace(
        go.Bar(
            x=labels,
            y=values,
            name=f"{number}부두",
            marker_color=port_colors[number],
            text=[
                f"{v:.2f}%" if pd.notna(v) else "자료 없음"
                for v in values
            ],
            textposition="outside",
            customdata=temp[["총작업시간_시간"]].to_numpy(),
            hovertemplate=(
                "선석: %{x}<br>"
                "평균 선석점유율: %{y:.2f}%<br>"
                "총 작업시간: %{customdata[0]:,.1f}시간"
                "<extra></extra>"
            )
        )
    )

# ======================================
# 5. 그래프 디자인
# ======================================

maximum = result["평균선석점유율_퍼센트"].max()

upper = (
    max(100, maximum * 1.15)
    if pd.notna(maximum)
    else 100
)

fig.update_layout(
    title=dict(
        text="2022~2025년 부두별·선석별 평균 선석점유율",
        x=0.5,
        font=dict(size=20)
    ),
    xaxis=dict(
        title="부두 / 선석",
        tickangle=-45
    ),
    yaxis=dict(
        title="평균 선석점유율 (%)",
        range=[0, upper],
        ticksuffix="%"
    ),
    barmode="group",
    template="plotly_white",
    width=1400,
    height=650,
    bargap=0.25,
    legend_title="부두"
)

# ======================================
# 6. PyCharm 노트북 내부에 출력
# ======================================

fig.show()

4부두: 오류·누락 1건 제외

[2022~2025년 부두별·선석별 평균 선석점유율]
 부두  선석  총작업시간_시간  평균선석점유율_퍼센트
  1   1  25490.57        72.70
  1   2  27742.68        79.12
  1   3  29446.03        83.98
  2   4  29982.72        85.51
  2   5  30264.97        86.31
  2   6  30925.38        88.20
  2   7  31080.73        88.64
  2   8  31347.87        89.40
  2   9     자료 없음        자료 없음
  3   1  29537.48        84.24
  3   2  21962.97        62.64
  3   3  21834.08        62.27
  3   4   8761.45        24.99
  4   1  20503.68        58.48
  4   2  26613.38        75.90
  4   3  27478.12        78.37
  4   4     자료 없음        자료 없음
  5   1  29422.40        83.91
  5   2  27962.50        79.75
  5   3   7677.90        21.90
  5   4  26947.87        76.85


# 부두별 선석별 월별 평균 선석점유율

In [25]:
import pandas as pd

# ======================================
# 1. CSV 불러오기
# ======================================

def load_dataset(filename):
    BASE_DIR = r'C:\work\Final_Project\data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'

    try:
        return pd.read_csv(TARGET_PATH, encoding='utf-8-sig')
    except FileNotFoundError:
        raise FileNotFoundError(
            f"파일을 찾을 수 없습니다: {TARGET_PATH}"
        )
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='cp949')


port1 = load_dataset('port1')
port2 = load_dataset('port2')
port3 = load_dataset('port3')
port4 = load_dataset('port4')
port5 = load_dataset('port5')

ports = [
    (1, port1),
    (2, port2),
    (3, port3),
    (4, port4),
    (5, port5)
]

# ======================================
# 2. 분석 대상 설정
# ======================================

# 모든 선석 포함
expected_berths = {
    1: [1, 2, 3],
    2: [4, 5, 6, 7, 8, 9],  # 9선석 포함
    3: [1, 2, 3, 4],
    4: [1, 2, 3, 4],        # 4선석 포함
    5: [1, 2, 3, 4]
}

START = pd.Timestamp("2022-01-01")
END = pd.Timestamp("2026-01-01")


# ======================================
# 3. 데이터 전처리
# ======================================

frames = []

for number, port in ports:

    data = port.copy()
    data.columns = data.columns.str.strip()

    # 접안시각 변환
    data["접안시각"] = pd.to_datetime(
        data["접안(예정)일시"],
        format="%Y-%m-%d %H:%M",
        errors="coerce"
    )

    # 출항시각 변환
    data["출항시각"] = pd.to_datetime(
        data["출항(예정)일시"],
        format="%Y-%m-%d %H:%M",
        errors="coerce"
    )

    # 선석 번호 변환
    data["선석"] = pd.to_numeric(
        data["선석"],
        errors="coerce"
    )

    # 오류 및 누락 데이터 제외
    invalid = (
        data[["접안시각", "출항시각", "선석"]]
        .isna().any(axis=1)
        | (data["출항시각"] <= data["접안시각"])
    )

    if invalid.any():
        print(f"{number}부두: 오류·누락 {invalid.sum()}건 제외")

    data = data.loc[~invalid].copy()

    # 해당 부두 선석만 선택
    data = data[
        data["선석"].isin(expected_berths[number])
    ].copy()

    # 분석 기간과 겹치는 기록만 선택
    data = data.loc[
        (data["접안시각"] < END)
        & (data["출항시각"] > START)
    ].copy()

    data["부두"] = number

    frames.append(
        data[[
            "부두",
            "선석",
            "접안시각",
            "출항시각"
        ]]
    )

detail = pd.concat(frames, ignore_index=True)


# ======================================
# 4. 월별·부두별·선석별 작업시간 계산
# ======================================

monthly_frames = []

months = pd.date_range(
    start=START,
    end=END,
    freq="MS",
    inclusive="left"
)

for month_start in months:

    month_end = month_start + pd.DateOffset(months=1)

    # 해당 월과 겹치는 기록 선택
    temp = detail.loc[
        (detail["접안시각"] < month_end)
        & (detail["출항시각"] > month_start)
    ].copy()

    if temp.empty:
        continue

    # 해당 월에 포함된 시간만 계산
    work_start = temp["접안시각"].clip(lower=month_start)
    work_end = temp["출항시각"].clip(upper=month_end)

    temp["작업시간_시간"] = (
        work_end - work_start
    ).dt.total_seconds() / 3600

    temp["기준월"] = month_start

    monthly_frames.append(
        temp[[
            "기준월",
            "부두",
            "선석",
            "작업시간_시간"
        ]]
    )


# ======================================
# 5. 월별 작업시간 합산
# ======================================

if monthly_frames:

    monthly_detail = pd.concat(
        monthly_frames,
        ignore_index=True
    )

    result = (
        monthly_detail.groupby(
            ["기준월", "부두", "선석"]
        )
        .agg(
            기록수=("작업시간_시간", "count"),
            총작업시간_시간=("작업시간_시간", "sum")
        )
    )

else:

    result = pd.DataFrame(
        columns=["기록수", "총작업시간_시간"]
    )

    result.index = pd.MultiIndex.from_arrays(
        [[], [], []],
        names=["기준월", "부두", "선석"]
    )


# ======================================
# 6. 모든 월·부두·선석 포함
# ======================================

expected_index = pd.MultiIndex.from_tuples(
    [
        (month, number, berth)
        for month in months
        for number, berths in expected_berths.items()
        for berth in berths
    ],
    names=["기준월", "부두", "선석"]
)

result = (
    result.reindex(expected_index)
    .reset_index()
)


# ======================================
# 7. 월별 선석점유율 계산
# ======================================

# 해당 월 총 시간
result["월총시간"] = (
    (
        result["기준월"] + pd.DateOffset(months=1)
    ) - result["기준월"]
).dt.total_seconds() / 3600

# 월별 선석점유율 계산
result["선석점유율_퍼센트"] = (
    result["총작업시간_시간"]
    / result["월총시간"]
    * 100
)

# 연도 및 월
result["연도"] = result["기준월"].dt.year
result["월"] = result["기준월"].dt.month

# 열 정리
result = result[
    [
        "연도",
        "월",
        "부두",
        "선석",
        "기록수",
        "총작업시간_시간",
        "월총시간",
        "선석점유율_퍼센트"
    ]
]

result = result.sort_values(
    ["연도", "월", "부두", "선석"]
).reset_index(drop=True)


# ======================================
# 8. 계산 결과 출력
# ======================================

print("\n[2022~2025년 월별·부두별·선석별 선석점유율]")

print(
    result.round(2).to_string(
        index=False,
        na_rep="자료 없음"
    )
)

4부두: 오류·누락 1건 제외

[2022~2025년 월별·부두별·선석별 선석점유율]
  연도  월  부두  선석   기록수  총작업시간_시간  월총시간  선석점유율_퍼센트
2022  1   1   1  27.0    638.00 744.0      85.75
2022  1   1   2  22.0    561.90 744.0      75.52
2022  1   1   3  22.0    692.70 744.0      93.10
2022  1   2   4  42.0    624.25 744.0      83.90
2022  1   2   5  36.0    622.50 744.0      83.67
2022  1   2   6  33.0    606.77 744.0      81.55
2022  1   2   7  28.0    685.58 744.0      92.15
2022  1   2   8  27.0    644.95 744.0      86.69
2022  1   2   9 자료 없음     자료 없음 744.0      자료 없음
2022  1   3   1  24.0    700.95 744.0      94.21
2022  1   3   2  29.0    540.00 744.0      72.58
2022  1   3   3  18.0    594.90 744.0      79.96
2022  1   3   4   8.0    153.00 744.0      20.56
2022  1   4   1  17.0    289.90 744.0      38.97
2022  1   4   2  24.0    542.43 744.0      72.91
2022  1   4   3  21.0    602.80 744.0      81.02
2022  1   4   4 자료 없음     자료 없음 744.0      자료 없음
2022  1   5   1  31.0    731.83 744.0      98.36
2022  1   5   2  21.0